In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('*******/20240929_4domain_54features.pkl')
frame_mfcc


In [ ]:
frame_mfcc = frame_mfcc[[
    'mfcc0', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7',
       'mfcc8', 'mfcc9', 'mfcc10', 'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14',
       'mfcc15', 'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20', 'mfcc21',
       'mfcc22', 'mfcc23', 'mfcc24', 'mfcc25', 'mfcc26', 'mfcc27', 'mfcc28',
       'mfcc29',

   'shimmerLocaldB_sma3nz_amean',
       'shimmerLocaldB_sma3nz_stddevNorm', 'loudness_sma3_amean',
      'loudness_sma3_stddevNorm', 'HNRdBACF_sma3nz_amean',
       'HNRdBACF_sma3nz_stddevNorm',

    'loudnessPeaksPerSec',
        'MeanVoicedSegmentLengthSec', 'StddevVoicedSegmentLengthSec',
       'MeanUnvoicedSegmentLength', 'StddevUnvoicedSegmentLength',
        'VoicedSegmentsPerSec',

    'F0semitoneFrom27.5Hz_sma3nz_amean',
    'F0semitoneFrom27.5Hz_sma3nz_stddevNorm', 'jitterLocal_sma3nz_amean',
    'jitterLocal_sma3nz_stddevNorm', 'F1frequency_sma3nz_amean',
    'F1frequency_sma3nz_stddevNorm', 'F2frequency_sma3nz_amean',
    'F2frequency_sma3nz_stddevNorm', 'F3frequency_sma3nz_stddevNorm',
    'F3amplitudeLogRelF0_sma3nz_amean', 'F1bandwidth_sma3nz_amean',
    'F1bandwidth_sma3nz_stddevNorm',

    'label', 'name'
                         ]]

In [ ]:
import numpy as np

X = frame_mfcc.iloc[:, :54]
X = X.to_numpy()


In [ ]:
new_features = np.expand_dims(X, axis=1)


In [ ]:
sample_labels= np.array(frame_mfcc['label'])
sample_labels

In [ ]:
import keras
from keras import regularizers

def locally_enhanced_ffn(inputs, ff_dim, dropout, kernel_sizeDepth, activation1, activation2):
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation=activation1)(inputs)

    x = layers.DepthwiseConv1D(kernel_size=kernel_sizeDepth,padding='same')(x)
    x = layers.Activation(activation2)(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1, activation=activation2)(x)

    x = layers.Add()([x, inputs]) 

    return x

# transformer_encoder 
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout,kernel_sizeDepth, activation1, activation2):
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs  

    x = locally_enhanced_ffn(res, ff_dim, dropout,kernel_sizeDepth, activation1, activation2)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return  x 


# build_model 
def build_model(
    input_shape,
    head_size,
    num_heads,
    ff_dim,
    num_transformer_blocks,
    dropout,
    kernel_sizeDepth,
    activation1,
    activation2,
        units
):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout,kernel_sizeDepth, activation1, activation2)

    # Bidirectional LSTM
    x = layers.Bidirectional(layers.LSTM(units=units, return_sequences=False))(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs)

In [ ]:
param_grid = {
 'head_size':  [2,],
    'num_heads':[2,],
    'ff_dim':  [ 64,],
    'num_transformer_blocks':  [ 2],
    'dropout':  [0.5],
    'units': [64],
    'kernel_sizeDepth':  [1],
    'activation1': ['relu'],
    'activation2': ['gelu'],
    'epochs': [ 500],
    'learning_rate' : [0.0001,],
}

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, roc_auc_score, f1_score, recall_score, confusion_matrix
from sklearn.utils import class_weight
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras import layers, Model, Input
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam, AdamW, RMSprop, Adagrad
import os

scaler = StandardScaler()
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

head_size = param_grid['head_size'][0]
num_heads = param_grid['num_heads'][0]
ff_dim = param_grid['ff_dim'][0]
num_transformer_blocks = param_grid['num_transformer_blocks'][0]
dropout = param_grid['dropout'][0]
units = param_grid['units'][0]
kernel_sizeDepth = param_grid['kernel_sizeDepth'][0]
activation1 = param_grid['activation1'][0]
activation2 = param_grid['activation2'][0]
epochs = param_grid['epochs'][0]
learning_rate = param_grid['learning_rate'][0]



x = 0


acc_arr = np.empty((10, 1))
f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))
balanced_accuracy_scores_arr = np.empty((10, 1))

fold_num = 1

for train_index, test_index in skf.split(new_features,sample_labels):


    X_train, X_test = new_features[train_index], new_features[test_index]
    y_train, y_test = sample_labels[train_index], sample_labels[test_index]

    X_train_reshaped = X_train.reshape(X_train.shape[0], X_train.shape[2])
    X_test_reshaped = X_test.reshape(X_test.shape[0], X_test.shape[2])

    X_train_scaled = scaler.fit_transform(X_train_reshaped)
    X_test_scaled = scaler.transform(X_test_reshaped)

    X_train = X_train_scaled.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])
    X_test = X_test_scaled.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])

    model = build_model(
    input_shape=(X_train.shape[1], X_train.shape[2]),
    head_size=head_size,
    num_heads=num_heads,
    ff_dim=ff_dim,
    num_transformer_blocks=num_transformer_blocks,
    dropout=dropout,
    units = units,
    kernel_sizeDepth=kernel_sizeDepth,
    activation1=activation1,
    activation2=activation2,
    )

    model.compile(
    loss= 'binary_crossentropy',
    metrics=[tf.keras.metrics.AUC()],

    )
    history = model.fit(
        X_train, y_train,
        epochs=epochs,  
        batch_size=32,
        verbose=0,
        )

    y_prob = model.predict(X_test)
    y_pred = (y_prob >= 0.5).astype(int)

    df_y_test = np.array(y_test)
    df_y_test = df_y_test.flatten('A')


    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)
    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_prob)
    balanced_accuracy_scores_arr[x] = metrics.balanced_accuracy_score(df_y_test, y_pred)

    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)

    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre

# save the model
    save_path = f'*********/model_fold{fold_num}.h5'

    folder_path = os.path.dirname(save_path)

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

# save the model
    model.save(save_path)

    fold_num = fold_num + 1

    x = x + 1

